### Step 1: Import Libraries

In [1]:
import pandas as pd

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

### Step 2: Load Dataset

In [9]:
df = pd.read_csv('./data/trainingandtestdata/training.1600000.processed.noemoticon.csv', encoding='latin-1', header=None)

In [10]:
df = df[[0, 5]]

In [11]:
df.columns = ['polarity', 'text']

In [12]:
df.head()

,polarity,text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,is upset that he can't update his Facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...
3,0,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all...."


### Step 3: Keep Only Positive and Negative Sentiments

In [13]:
df['polarity'] = df['polarity'].map({0:0, 4:1}) # 0: negative 1:positive

In [14]:
df['polarity'].value_counts()

polarity
0    800000
1    800000
Name: count, dtype: int64

### Step 4: Clean the Tweets

In [15]:
df['clean_text'] = df['text'].apply(lambda x: x.lower())

In [16]:
df.head()

,polarity,text,clean_text
0,0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...","@switchfoot http://twitpic.com/2y1zl - awww, t..."
1,0,is upset that he can't update his Facebook by ...,is upset that he can't update his facebook by ...
2,0,@Kenichan I dived many times for the ball. Man...,@kenichan i dived many times for the ball. man...
3,0,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire
4,0,"@nationwideclass no, it's not behaving at all....","@nationwideclass no, it's not behaving at all...."


### Step 5: Train Test Split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['polarity'],
    test_size=0.2,
    random_state=42
)

In [18]:
print('Train size:', len(X_train))
print('Test size:', len(X_test))

Train size: 1280000
Test size: 320000


### Step 6: Perform Vectorization

In [19]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

In [20]:
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [21]:
print("TF-IDF shape (train):", X_train_tfidf.shape)
print("TF-IDF shape (test):", X_test_tfidf.shape)

TF-IDF shape (train): (1280000, 5000)
TF-IDF shape (test): (320000, 5000)


### Step 7: Train Bernoulli Naive Bayes Model

In [22]:
bnb = BernoulliNB()

In [23]:
bnb.fit(X_train_tfidf, y_train)

,alpha,1.0
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [24]:
bnb_pred = bnb.predict(X_test_tfidf)

In [25]:
print('bernoulli Naive Bayes Accuracy:', accuracy_score(y_test, bnb_pred))

bernoulli Naive Bayes Accuracy: 0.766490625


In [28]:
print('BernoulliNB Classification Report:\n', classification_report(y_test, bnb_pred))

BernoulliNB Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.75      0.76    159494
           1       0.76      0.78      0.77    160506

    accuracy                           0.77    320000
   macro avg       0.77      0.77      0.77    320000
weighted avg       0.77      0.77      0.77    320000



### Step 8: Train Support Vector Machine (SVM) Model

In [29]:
svm = LinearSVC()

In [30]:
svm.fit(X_train_tfidf, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [31]:
svm_pred = svm.predict(X_test_tfidf)

In [32]:
print("SVM Accuracy:", accuracy_score(y_test, svm_pred))

SVM Accuracy: 0.795321875


In [33]:
print("SVM Classification Report:\n", classification_report(y_test, svm_pred))

SVM Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.78      0.79    159494
           1       0.79      0.81      0.80    160506

    accuracy                           0.80    320000
   macro avg       0.80      0.80      0.80    320000
weighted avg       0.80      0.80      0.80    320000



### Step 9: Train Logistic Regression Model

In [34]:
logreg = LogisticRegression()

In [35]:
logreg.fit(X_train_tfidf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [36]:
logreg_pred = logreg.predict(X_test_tfidf)

In [37]:
print("Logistic Regression Accuracy:", accuracy_score(y_test, logreg_pred))

Logistic Regression Accuracy: 0.796084375


In [38]:
print("Logistic Regression Classification Report:\n", classification_report(y_test, logreg_pred))

Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.78      0.79    159494
           1       0.79      0.81      0.80    160506

    accuracy                           0.80    320000
   macro avg       0.80      0.80      0.80    320000
weighted avg       0.80      0.80      0.80    320000



### Step 10: Make Predictions on Sample Tweets

In [39]:
sample_tweets = ["I love this!", "I hate that!", "It was okay, not great."]

In [40]:
sample_vec = vectorizer.transform(sample_tweets)

In [41]:
print("Sample Predictions:")
print("\nBernoulliNB:", bnb.predict(sample_vec))
print("\nSVM:", svm.predict(sample_vec))
print("\nLogistic Regression:", logreg.predict(sample_vec))

Sample Predictions:

BernoulliNB: [1 0 1]

SVM: [1 0 1]

Logistic Regression: [1 0 1]
